In [1]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
start_time = Time("2025-01-01 00:00:00")
end_time = Time("2025-01-01 04:00:00")
control_slice_duration = np.timedelta64(int(60*1e6), "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

controller = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_num=0,
    # azimuth_range=None,
    # elevation_range=None,
)

In [ ]:
fig = controller.plot()
fig.show()

In [ ]:
r = controller.generate()

df = pd.DataFrame({"x": ecefs[0], "y": ecefs[1], "z": ecefs[2]})
fig_point = px.scatter_3d(
    df, x="x", y="y", z="z", color_discrete_sequence=["red"], animation_frame=df.index
)
fig_point.update_traces(marker_size=3)
fig_line = px.line_3d(df, x="x", y="y", z="z")

# use `fig_point` as base and merge `fig_line` into it
fig = fig_point
for tr in fig_line.select_traces():
    fig_point.add_trace(tr)

# enable the button to toggle project lines to the grid
# ref:
# - https://github.com/plotly/plotly.py/blob/ae85438086c325a32d91604ae00893f85f125cdc/CHANGELOG.md?plain=1#L480
# - https://github.com/plotly/plotly.py/issues/3274
fig.show(config={
    "modeBarButtonsToAdd": ["v1hovermode", "toggleSpikeLines"],
})